# Embedding view of the dual mandate

Same corpus, different representation. Documents are embedded with a local Ollama model and compared to two short mandate prototypes.

In [ ]:
import os
import re
import tempfile
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "fomc_nlp_matplotlib"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 140)

ROOT = Path.cwd()
if not (ROOT / "data" / "raw" / "fomc_documents_raw.csv").exists():
    ROOT = ROOT.parent

FIGURES = ROOT / "figures"
FIGURES.mkdir(exist_ok=True)

OLLAMA_URL = os.getenv("OLLAMA_URL", "http://127.0.0.1:11434")
EMBED_MODEL = os.getenv("OLLAMA_EMBED_MODEL", "nomic-embed-text")

This uses the local Ollama model `nomic-embed-text`.

In [ ]:
def ollama_embed_one(text, model=EMBED_MODEL):
    words = str(text).split()
    for limit in [450, 250, 120]:
        prompt = " ".join(words[:limit])
        response = requests.post(
            f"{OLLAMA_URL}/api/embeddings",
            json={"model": model, "prompt": prompt},
            timeout=120,
        )
        if response.ok:
            return response.json()["embedding"]
    response.raise_for_status()


def ollama_embed_texts(texts, model=EMBED_MODEL):
    vectors = [ollama_embed_one(text, model=model) for text in texts]
    return normalize(np.asarray(vectors, dtype="float32"))


test_vector = ollama_embed_texts(["inflation and labor market"])
test_vector.shape

In [ ]:
raw = pd.read_csv(ROOT / "data" / "raw" / "fomc_documents_raw.csv", parse_dates=["date"])

TOKEN_RE = re.compile(r"[a-z]+(?:'[a-z]+)?", re.IGNORECASE)


def clean_text(text):
    text = "" if pd.isna(text) else str(text)
    text = text.replace(" ", " ")
    text = text.replace("’", "'").replace("‘", "'")
    text = text.replace("“", '"').replace("”", '"')
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def count_words(text):
    return len(TOKEN_RE.findall(clean_text(text)))


def strip_fed_boilerplate(text):
    text = "" if pd.isna(text) else str(text)
    lower = text.lower()
    starts = [
        lower.find("for immediate release"),
        lower.find("minutes of the federal open market committee"),
    ]
    starts = [idx for idx in starts if idx >= 0]
    if starts:
        text = text[min(starts):]
        lower = text.lower()
    for marker in ["return to top", "home | fomc", "last update:", "to comment on this site"]:
        idx = lower.find(marker)
        if idx > 100:
            text = text[:idx]
            break
    return text


def excerpt_across_document(text, max_words=450):
    words = clean_text(strip_fed_boilerplate(text)).split()
    if len(words) <= max_words:
        return " ".join(words)
    part = max_words // 3
    middle = len(words) // 2
    selected = words[:part] + words[middle:middle + part] + words[-part:]
    return " ".join(selected)


df = (
    raw.sort_values("date")
    .reset_index(drop=True)
    .assign(
        statement_clean_text=lambda x: x["statement_text"].map(clean_text),
        minutes_clean_text=lambda x: x["minutes_text"].map(clean_text),
        statement_n_words=lambda x: x["statement_text"].map(count_words),
        minutes_n_words=lambda x: x["minutes_text"].map(count_words),
        statement_embed_text=lambda x: x["statement_text"].map(excerpt_across_document),
        minutes_embed_text=lambda x: x["minutes_text"].map(excerpt_across_document),
    )
)

df[["date", "statement_n_words", "minutes_n_words"]].head()

Minutes are too long for a clean one-shot embedding. I keep a short beginning / middle / end excerpt with the same word budget for every document.

In [ ]:
INFLATION_TERMS = [
    "inflation", "inflationary", "price stability", "price pressures",
    "inflation expectations", "core inflation", "pce inflation", "consumer prices",
    "energy prices", "food prices", "cost pressures", "supply constraints", "disinflation",
]

LABOR_TERMS = [
    "employment", "unemployment", "labor market", "job gains", "payrolls",
    "hiring", "layoffs", "wages", "labor demand", "labor supply",
    "labor force", "slack", "participation", "vacancies", "job openings",
]


def count_terms(text, terms):
    text = clean_text(text)
    total = 0
    for term in terms:
        pattern = re.escape(term.lower()).replace(r"\ ", r"\s+")
        total += len(re.findall(rf"\b{pattern}\b", text))
    return total


def add_topic_score(data, corpus, topic, terms):
    text_col = f"{corpus}_clean_text"
    n_col = f"{corpus}_n_words"
    count_col = f"{corpus}_{topic}_count"
    freq_col = f"{corpus}_{topic}_per_1000"
    counts = data[text_col].map(lambda text: count_terms(text, terms))
    return data.assign(**{
        count_col: counts,
        freq_col: lambda x: 1_000 * x[count_col] / x[n_col].replace(0, np.nan),
    })


df = (
    df.pipe(add_topic_score, "statement", "inflation", INFLATION_TERMS)
    .pipe(add_topic_score, "statement", "labor", LABOR_TERMS)
    .pipe(add_topic_score, "minutes", "inflation", INFLATION_TERMS)
    .pipe(add_topic_score, "minutes", "labor", LABOR_TERMS)
    .assign(
        statement_dictionary_balance=lambda x: x["statement_inflation_per_1000"] - x["statement_labor_per_1000"],
        minutes_dictionary_balance=lambda x: x["minutes_inflation_per_1000"] - x["minutes_labor_per_1000"],
    )
)

In [ ]:
statement_docs = (
    df.loc[:, ["date", "year", "statement_embed_text", "statement_dictionary_balance"]]
    .rename(columns={"statement_embed_text": "text", "statement_dictionary_balance": "dictionary_balance"})
    .assign(document_type="statement")
)
minutes_docs = (
    df.loc[:, ["date", "year", "minutes_embed_text", "minutes_dictionary_balance"]]
    .rename(columns={"minutes_embed_text": "text", "minutes_dictionary_balance": "dictionary_balance"})
    .assign(document_type="minutes")
)

docs = pd.concat([statement_docs, minutes_docs], ignore_index=True)
docs.head()

Prototype trick: embed one short inflation phrase and one short labor-market phrase, then compare cosine similarities.

In [ ]:
INFLATION_PROTOTYPE = "inflation pressure, price stability, rising prices, inflation expectations, cost pressures"
LABOR_PROTOTYPE = "maximum employment, unemployment, labor market slack, job losses, weak payroll growth"

texts = docs["text"].tolist() + [INFLATION_PROTOTYPE, LABOR_PROTOTYPE]
embeddings = ollama_embed_texts(texts)

doc_vectors = embeddings[:-2]
inflation_vector = embeddings[-2]
labor_vector = embeddings[-1]

docs = docs.assign(
    inflation_similarity=doc_vectors @ inflation_vector,
    labor_similarity=doc_vectors @ labor_vector,
)

docs = docs.assign(embedding_balance=lambda x: x["inflation_similarity"] - x["labor_similarity"])

docs[["document_type", "dictionary_balance", "embedding_balance"]].describe().round(3)

In [ ]:
(
    docs.groupby("document_type")[["dictionary_balance", "embedding_balance", "inflation_similarity", "labor_similarity"]]
    .mean()
    .round(3)
)

In [ ]:
(
    docs.groupby("document_type")[["dictionary_balance", "embedding_balance"]]
    .corr(method="spearman")
    .loc[(slice(None), "dictionary_balance"), "embedding_balance"]
    .round(3)
)

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 4.4))
sns.scatterplot(
    data=docs,
    x="dictionary_balance",
    y="embedding_balance",
    hue="document_type",
    alpha=0.75,
    ax=ax,
)
ax.axhline(0, color="black", linewidth=0.8)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Dictionary balance vs embedding balance")
ax.set_xlabel("dictionary balance")
ax.set_ylabel("prototype similarity balance")
fig.tight_layout()
fig.savefig(FIGURES / "embedding_dictionary_balance.png", bbox_inches="tight")

In [ ]:
yearly_embedding = (
    docs.groupby(["year", "document_type"], as_index=False)["embedding_balance"]
    .mean()
)

fig, ax = plt.subplots(figsize=(8.0, 3.6))
sns.lineplot(data=yearly_embedding, x="year", y="embedding_balance", hue="document_type", ax=ax)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Embedding balance over time")
ax.set_xlabel("")
ax.set_ylabel("inflation similarity minus labor similarity")
fig.tight_layout()
fig.savefig(FIGURES / "embedding_balance_over_time.png", bbox_inches="tight")

A quick PCA view shows the largest directions in the embedding space.

In [ ]:
pca = PCA(n_components=2, random_state=42)
coords = docs.assign(**dict(zip(["pc1", "pc2"], pca.fit_transform(doc_vectors).T)))

fig, ax = plt.subplots(figsize=(7.0, 5.0))
norm = plt.Normalize(coords["embedding_balance"].min(), coords["embedding_balance"].max())
cmap = plt.cm.coolwarm
for document_type, marker in [("statement", "o"), ("minutes", "X")]:
    plot_data = coords.loc[coords["document_type"].eq(document_type)]
    ax.scatter(
        plot_data["pc1"],
        plot_data["pc2"],
        c=plot_data["embedding_balance"],
        cmap=cmap,
        norm=norm,
        marker=marker,
        s=42,
        alpha=0.85,
        label=document_type,
        edgecolor="white",
        linewidth=0.4,
    )
fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax, label="embedding balance")
ax.set_title("Ollama embedding PCA map")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
ax.legend(title="document type")
fig.tight_layout()
fig.savefig(FIGURES / "embedding_pca_map.png", bbox_inches="tight")

pd.Series({
    "pc1_explained": pca.explained_variance_ratio_[0],
    "pc2_explained": pca.explained_variance_ratio_[1],
}).round(3)

The embedding balance gives a second reading of the same inflation-versus-labor question.